<a href="https://colab.research.google.com/github/jhenningsen/Equity_Analysis/blob/main/MomentumRotation/MomentumRotation_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### SMA-Based Daily Recommendation Script

This script is structured to fulfill your requirements. It focuses on generating a daily 'buy' recommendation for a single ETF based on a simple SMA-crossing strategy. It doesn't execute trades but prints the recommended ticker, simulating the decision-making process.

**Key Features:**

*   **Rule-Based Trading**: The core logic is encapsulated in the `generate_recommendation` method, scheduled to run daily before market close.
*   **Warm-up Periods**: Ensures that all SMAs have enough historical data before recommendations begin.
*   **Universe Definition**: Starts with a predefined list of 6 commonly traded ETFs (SPY, QQQ, DIA, IWM, GLD, TLT).
*   **Indicator-Driven**: Utilizes 3, 5, 10, and 20-day Simple Moving Averages (SMAs).
*   **Concentrated Positions**: Identifies and recommends only one best-performing ETF each day based on the SMA criteria.

**Recommendation Logic:**

For each ETF, a 'score' is calculated. This score increases if:
1.  The current price is above its 3-day SMA.
2.  The 3-day SMA is above the 5-day SMA.
3.  The 5-day SMA is above the 10-day SMA.
4.  The 10-day SMA is above the 20-day SMA.

The ETF with the highest score (indicating a strong upward trend across multiple timeframes) is selected as the daily recommendation. If multiple ETFs have the same highest score, the first one encountered will be chosen.

In [129]:
# Setup and SMABasedRecommendationColab Class Definition

# Install yfinance if you haven't already
!pip install yfinance

import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Define the start and end dates for the simulation
USER_START_DATE = '2020-01-01'
USER_END_DATE = '2026-01-01' # This date will be inclusive in the simulation

class SMABasedRecommendationColab:

    def __init__(self, tickers=None, sma_periods=None, start_date=None, end_date=None):
        # === CONFIGURATION ===
        # self.universe_tickers = tickers if tickers is not None else ["SPY", "QQQ", "DIA", "IWM", "GLD", "TLT"]
        # self.universe_tickers = tickers if tickers is not None else ["QQQ", "SPY", "DIA", "IWM", "GLD", "TLT"]
        self.universe_tickers = tickers if tickers is not None else ["QQQ", "PSQ", "SPY", "SH"]
        self.sma_periods = sma_periods if sma_periods is not None else [3, 5, 10, 20]
        self.start_date = pd.to_datetime(start_date) if start_date else pd.to_datetime(USER_START_DATE)
        self.end_date = pd.to_datetime(end_date) if end_date else pd.to_datetime(USER_END_DATE)

        # === DATA STORAGE ===
        self.historical_data = pd.DataFrame() # To store all downloaded price data
        self.smas_data = {} # To store SMA values for each ticker
        self.recommendations = [] # To store daily recommendations

        # === WARM-UP ===
        # We need enough data to calculate the longest SMA
        self.warmup_days = max(self.sma_periods) + 5 # +5 days for buffer

        print("Initializing data download...")
        self._download_data()
        print("Data download complete. Calculating SMAs...")
        self._calculate_all_smas()
        print("SMAs calculated. Ready for simulation.")


    def _download_data(self):
        # Download data for all tickers
        # We need data starting from before the actual start_date due to warm-up
        data_start = self.start_date - timedelta(days=self.warmup_days * 2) # Get extra data for safety
        # Setting `auto_adjust=False` to use unadjusted close prices.
        self.historical_data = yf.download(self.universe_tickers, start=data_start, end=self.end_date + timedelta(days=1), interval='1d', auto_adjust=False)['Close']

        # Ensure we have enough data for the entire period after warm-up
        min_data_points = max(self.sma_periods)
        self.historical_data = self.historical_data.dropna(axis=1, thresh=min_data_points) # Drop tickers with insufficient data
        self.universe_tickers = self.historical_data.columns.tolist() # Update tickers based on available data
        if not self.universe_tickers:
            raise ValueError("No sufficient data downloaded for any ticker in the universe.")

    def _calculate_all_smas(self):
        for ticker in self.universe_tickers:
            self.smas_data[ticker] = pd.DataFrame(index=self.historical_data.index)
            for period in self.sma_periods:
                self.smas_data[ticker][f'SMA_{period}'] = self.historical_data[ticker].rolling(window=period).mean()

    def _generate_daily_recommendation(self, current_date):
        candidate_scores = {}

        for ticker in self.universe_tickers:
            # Get current price and SMA values for the specific date
            current_price_series = self.historical_data.loc[current_date, ticker]
            if pd.isna(current_price_series): # Skip if no data for this date
                continue
            current_price = current_price_series

            sma_values = {}
            smas_ready = True
            for period in self.sma_periods:
                sma_series = self.smas_data[ticker].loc[current_date, f'SMA_{period}']
                if pd.isna(sma_series):
                    smas_ready = False
                    break
                sma_values[period] = sma_series

            if not smas_ready:
                # print(f"DEBUG: SMAs for {ticker} not ready on {current_date.strftime('%Y-%m-%d')}. Skipping.")
                continue

            # Score based on SMA hierarchy: price > SMA3 > SMA5 > SMA10 > SMA20
            score = 0
            if current_price > sma_values[3]:
                score += 1
            if sma_values[3] > sma_values[5]:
                score += 1
            if sma_values[5] > sma_values[10]:
                score += 1
            if sma_values[10] > sma_values[20]:
                score += 1

            candidate_scores[ticker] = score

        if not candidate_scores:
            return None, 0, candidate_scores # Return all scores even if no best ticker

        # Find the ticker with the highest score
        best_ticker = max(candidate_scores.items(), key=lambda item: item[1])[0]
        highest_score = candidate_scores[best_ticker]

        return best_ticker, highest_score, candidate_scores

    def run_simulation(self):
        trade_dates = self.historical_data.index[self.historical_data.index >= self.start_date].normalize().unique()
        warmup_end_date = self.start_date + timedelta(days=self.warmup_days)

        current_held_ticker = None # Ticker currently held by the strategy
        current_held_score = -1 # Score of the currently held ticker when it was decided

        for current_date in trade_dates:
            if current_date < self.start_date: # Ensure we only process from the actual start date
                continue
            if current_date < warmup_end_date: # Simulate warm-up period
                continue
            if current_date > self.end_date: # Ensure simulation does not go beyond the user-defined end date
                continue

            # QuantConnect-like scheduling: run on trading days
            if current_date.weekday() >= 5: # Skip weekends
                 continue

            # Check if market is open (i.e., data is available for the date)
            if current_date not in self.historical_data.index:
                continue

            # Get the raw recommendation from the scoring algorithm
            raw_recommended_ticker, raw_highest_score, all_scores_on_today = self._generate_daily_recommendation(current_date)

            decided_ticker_for_today = current_held_ticker # Start by assuming we maintain position
            decided_score_for_today = current_held_score

            # Get the score of the current holding on TODAY's date (if any)
            score_of_current_holding_on_today = -1
            if current_held_ticker is not None and current_held_ticker in all_scores_on_today:
                score_of_current_holding_on_today = all_scores_on_today[current_held_ticker]

            # Get the score of the raw recommended ticker on TODAY's date (if any)
            score_of_raw_recommended_on_today = -1
            if raw_recommended_ticker is not None and raw_recommended_ticker != 'NONE' and raw_recommended_ticker in all_scores_on_today:
                score_of_raw_recommended_on_today = all_scores_on_today[raw_recommended_ticker]

            # Apply the trading decision logic
            if current_held_ticker is None: # We are not holding anything
                if raw_recommended_ticker != 'NONE': # If there's a new recommendation, take it
                    decided_ticker_for_today = raw_recommended_ticker
                    decided_score_for_today = raw_highest_score # Use the score from the raw recommendation
                else:
                    decided_ticker_for_today = 'NONE'
                    decided_score_for_today = 0
            elif raw_recommended_ticker == 'NONE': # No recommendation, so exit current position
                decided_ticker_for_today = 'NONE'
                decided_score_for_today = 0
            elif raw_recommended_ticker == current_held_ticker: # Recommended ticker is the same as currently held
                decided_ticker_for_today = current_held_ticker
                decided_score_for_today = score_of_current_holding_on_today # Update score with today's score
            else: # Raw recommended ticker is different from currently held
                # Switch only if the new recommendation is *strictly higher* than current holding's score today
                if score_of_raw_recommended_on_today > score_of_current_holding_on_today:
                    decided_ticker_for_today = raw_recommended_ticker
                    decided_score_for_today = score_of_raw_recommended_on_today
                else:
                    # Maintain current position if new recommendation is not strictly better
                    decided_ticker_for_today = current_held_ticker
                    decided_score_for_today = score_of_current_holding_on_today

            # Update current holding for the next day's evaluation
            current_held_ticker = decided_ticker_for_today
            current_held_score = decided_score_for_today

            # Store the finalized decision
            rec_close_price = np.nan
            if decided_ticker_for_today != 'NONE' and decided_ticker_for_today in self.historical_data.columns:
                rec_close_price = self.historical_data.loc[current_date, decided_ticker_for_today]

            self.recommendations.append({
                'date': current_date.strftime('%Y-%m-%d'),
                'recommended_ticker': decided_ticker_for_today, # This is now the *finalized* decision
                'score': decided_score_for_today, # This is now the *finalized* score for the decision
                'all_scores': all_scores_on_today, # Still store all raw scores for debugging/analysis
                'close_price': rec_close_price # Close price of the *decided* ticker
            })

            print(f"[{current_date.strftime('%Y-%m-%d')}] Decision: {decided_ticker_for_today} (Score: {decided_score_for_today}) | All Raw Scores: {all_scores_on_today})")

        print("\nSimulation Complete.")

# Instantiate the recommendation engine
# You can customize the tickers and SMA periods here.
rec_engine = SMABasedRecommendationColab(start_date=USER_START_DATE, end_date=USER_END_DATE)

# Run the simulation to generate daily recommendations
rec_engine.run_simulation()

# Access recommendations and display the first few rows
df_recommendations = pd.DataFrame(rec_engine.recommendations)
print("\nGenerated Recommendations (first 5 rows):")
display(df_recommendations)

[**********************50%                       ]  2 of 4 completed

Initializing data download...


[*********************100%***********************]  4 of 4 completed


Data download complete. Calculating SMAs...
SMAs calculated. Ready for simulation.
[2020-01-27] Decision: SH (Score: 3) | All Raw Scores: {'PSQ': 2, 'QQQ': 2, 'SH': 3, 'SPY': 2})
[2020-01-28] Decision: QQQ (Score: 3) | All Raw Scores: {'PSQ': 1, 'QQQ': 3, 'SH': 2, 'SPY': 2})
[2020-01-29] Decision: QQQ (Score: 2) | All Raw Scores: {'PSQ': 2, 'QQQ': 2, 'SH': 2, 'SPY': 2})
[2020-01-30] Decision: QQQ (Score: 3) | All Raw Scores: {'PSQ': 1, 'QQQ': 3, 'SH': 1, 'SPY': 3})
[2020-01-31] Decision: SH (Score: 3) | All Raw Scores: {'PSQ': 2, 'QQQ': 2, 'SH': 3, 'SPY': 2})
[2020-02-03] Decision: SH (Score: 3) | All Raw Scores: {'PSQ': 2, 'QQQ': 2, 'SH': 3, 'SPY': 1})
[2020-02-04] Decision: QQQ (Score: 4) | All Raw Scores: {'PSQ': 0, 'QQQ': 4, 'SH': 3, 'SPY': 1})
[2020-02-05] Decision: QQQ (Score: 4) | All Raw Scores: {'PSQ': 0, 'QQQ': 4, 'SH': 2, 'SPY': 2})
[2020-02-06] Decision: QQQ (Score: 4) | All Raw Scores: {'PSQ': 0, 'QQQ': 4, 'SH': 1, 'SPY': 3})
[2020-02-07] Decision: QQQ (Score: 3) | All Raw

,date,recommended_ticker,score,all_scores,close_price
0,2020-01-27,SH,3,"{'PSQ': 2, 'QQQ': 2, 'SH': 3, 'SPY': 2}",95.599998
1,2020-01-28,QQQ,3,"{'PSQ': 1, 'QQQ': 3, 'SH': 2, 'SPY': 2}",221.449997
2,2020-01-29,QQQ,2,"{'PSQ': 2, 'QQQ': 2, 'SH': 2, 'SPY': 2}",221.809998
3,2020-01-30,QQQ,3,"{'PSQ': 1, 'QQQ': 3, 'SH': 1, 'SPY': 3}",222.600006
4,2020-01-31,SH,3,"{'PSQ': 2, 'QQQ': 2, 'SH': 3, 'SPY': 2}",96.199997
...,...,...,...,...,...
1487,2025-12-24,SPY,3,"{'PSQ': 1, 'QQQ': 3, 'SH': 1, 'SPY': 3}",690.380005
1488,2025-12-26,SPY,3,"{'PSQ': 1, 'QQQ': 3, 'SH': 1, 'SPY': 3}",690.309998
1489,2025-12-29,SPY,2,"{'PSQ': 2, 'QQQ': 2, 'SH': 1, 'SPY': 2}",687.849976
1490,2025-12-30,PSQ,2,"{'PSQ': 2, 'QQQ': 1, 'SH': 1, 'SPY': 1}",29.940001


In [130]:
# Add Close Prices to Recommendations DataFrame

close_prices = []
for index, row in df_recommendations.iterrows():
    date = pd.to_datetime(row['date'])
    ticker = row['recommended_ticker']
    if ticker != 'NONE' and date in rec_engine.historical_data.index:
        try:
            close_price = rec_engine.historical_data.loc[date, ticker]
            close_prices.append(close_price)
        except KeyError:
            close_prices.append(np.nan) # Append NaN if data is missing for some reason
    else:
        close_prices.append(np.nan)

df_recommendations['close_price'] = close_prices
print("\nRecommendations with Close Price (first 5 rows):")
display(df_recommendations)


Recommendations with Close Price (first 5 rows):


,date,recommended_ticker,score,all_scores,close_price
0,2020-01-27,SH,3,"{'PSQ': 2, 'QQQ': 2, 'SH': 3, 'SPY': 2}",95.599998
1,2020-01-28,QQQ,3,"{'PSQ': 1, 'QQQ': 3, 'SH': 2, 'SPY': 2}",221.449997
2,2020-01-29,QQQ,2,"{'PSQ': 2, 'QQQ': 2, 'SH': 2, 'SPY': 2}",221.809998
3,2020-01-30,QQQ,3,"{'PSQ': 1, 'QQQ': 3, 'SH': 1, 'SPY': 3}",222.600006
4,2020-01-31,SH,3,"{'PSQ': 2, 'QQQ': 2, 'SH': 3, 'SPY': 2}",96.199997
...,...,...,...,...,...
1487,2025-12-24,SPY,3,"{'PSQ': 1, 'QQQ': 3, 'SH': 1, 'SPY': 3}",690.380005
1488,2025-12-26,SPY,3,"{'PSQ': 1, 'QQQ': 3, 'SH': 1, 'SPY': 3}",690.309998
1489,2025-12-29,SPY,2,"{'PSQ': 2, 'QQQ': 2, 'SH': 1, 'SPY': 2}",687.849976
1490,2025-12-30,PSQ,2,"{'PSQ': 2, 'QQQ': 1, 'SH': 1, 'SPY': 1}",29.940001


In [131]:
# Simulate Trades and Calculate Profits
investment_amount = 5000

trades_list = []
current_trade_start_index = -1
current_trade_ticker = None # Ticker currently held
current_entry_score = -1 # Score of the ticker when it was entered

for i in range(len(df_recommendations)):
    row = df_recommendations.iloc[i]
    current_date = pd.to_datetime(row['date'])
    # The 'recommended_ticker' and 'score' from df_recommendations are from the class's default tie-breaking
    class_recommended_ticker = row['recommended_ticker']
    class_recommended_score = row['score']
    all_scores_on_today = row['all_scores']

    # --- Determine the actual ticker to hold for this day based on new rules ---
    actual_ticker_to_hold = current_trade_ticker # Assume we hold what we have initially

    # Get the score of the current holding on TODAY's date
    score_of_current_holding_on_today = -1
    if current_trade_ticker is not None and current_trade_ticker in all_scores_on_today:
        score_of_current_holding_on_today = all_scores_on_today[current_trade_ticker]

    # Get the score of the class's recommended ticker on TODAY's date
    score_of_class_recommended_on_today = -1
    if class_recommended_ticker != 'NONE' and class_recommended_ticker in all_scores_on_today:
        score_of_class_recommended_on_today = all_scores_on_today[class_recommended_ticker]

    should_make_a_new_trade = False # Flag to indicate if we are BUYING or SELLING (entering/exiting)

    if current_trade_ticker is None: # We are not holding anything
        if class_recommended_ticker != 'NONE':
            # Start a new trade with the class's recommendation
            actual_ticker_to_hold = class_recommended_ticker
            should_make_a_new_trade = True
    else: # We are currently holding a ticker (current_trade_ticker is not None)
        if class_recommended_ticker == 'NONE':
            # Class recommends nothing, so we must exit our current position
            actual_ticker_to_hold = 'NONE'
            should_make_a_new_trade = True
        elif class_recommended_ticker == current_trade_ticker:
            # Class recommends what we already hold. No trade needed.
            # actual_ticker_to_hold is already set to current_trade_ticker
            pass
        else: # Class recommends a different ticker (class_recommended_ticker != current_trade_ticker)
            # Compare the score of the new recommendation against our current holding's score *on this day*
            if score_of_class_recommended_on_today > score_of_current_holding_on_today:
                # The new recommendation is strictly better, so we switch
                actual_ticker_to_hold = class_recommended_ticker
                should_make_a_new_trade = True
            # Else (score_of_class_recommended_on_today <= score_of_current_holding_on_today):
            # The new recommendation is NOT strictly better. We stick with our current holding.
            # actual_ticker_to_hold is already set to current_trade_ticker

    # Now, execute the trade action if should_make_a_new_trade is True
    if should_make_a_new_trade:
        # If we were holding something, close the old trade
        if current_trade_ticker is not None:
            entry_row = df_recommendations.iloc[current_trade_start_index]
            entry_ticker = current_trade_ticker # This was the ticker chosen at entry
            entry_date = pd.to_datetime(entry_row['date'])
            entry_price = entry_row['close_price']
            entry_score = current_entry_score # Use the score when it was *actually* entered

            exit_date = current_date # Exit today
            # Fetch exit price of the ticker being exited
            if entry_ticker in rec_engine.historical_data.columns and exit_date in rec_engine.historical_data.index:
                exit_price = rec_engine.historical_data.loc[exit_date, entry_ticker]
            else:
                exit_price = np.nan

            if not pd.isna(entry_price) and not pd.isna(exit_price) and entry_price > 0:
                num_shares = np.floor(investment_amount / entry_price)
                profit = (exit_price - entry_price) * num_shares
                trades_list.append({
                    'ticker': entry_ticker,
                    'entry_date': entry_date,
                    'entry_price': entry_price,
                    'exit_date': exit_date,
                    'exit_price': exit_price,
                    'num_shares': num_shares,
                    'profit': profit,
                    'holding_days': (exit_date - entry_date).days,
                    'entry_score': entry_score,
                    'exit_score': score_of_class_recommended_on_today if actual_ticker_to_hold != 'NONE' else 0 # Score of the *new* ticker if switching, or 0 if exiting to NONE
                })

        # Update current holding based on the decision
        if actual_ticker_to_hold != 'NONE':
            current_trade_ticker = actual_ticker_to_hold
            current_trade_start_index = i
            # Store the score of this ticker when it was entered
            current_entry_score = all_scores_on_today.get(current_trade_ticker, 0) # Get score for the ticker we are *now* holding
        else: # If we are exiting to NONE
            current_trade_ticker = None
            current_trade_start_index = -1
            current_entry_score = -1 # Reset entry score

# After the loop, if there's an open trade, close it at the last available day
if current_trade_ticker is not None and current_trade_start_index != -1:
    entry_row = df_recommendations.iloc[current_trade_start_index]
    entry_ticker = current_trade_ticker # The ticker that was actually held
    entry_date = pd.to_datetime(entry_row['date'])
    entry_price = entry_row['close_price']
    entry_score = current_entry_score # Score when the trade was entered

    # The exit_date for the open trade is the last day of the simulation
    exit_row = df_recommendations.iloc[len(df_recommendations) - 1] # Last day of the simulation
    exit_date = pd.to_datetime(exit_row['date'])
    # Fetch the actual close price of the ticker that is being exited from historical_data
    if entry_ticker in rec_engine.historical_data.columns and exit_date in rec_engine.historical_data.index:
        exit_price = rec_engine.historical_data.loc[exit_date, entry_ticker]
    else:
        exit_price = np.nan # Handle cases where data might be missing

    if not pd.isna(entry_price) and not pd.isna(exit_price) and entry_price > 0:
        num_shares = np.floor(investment_amount / entry_price)
        profit = (exit_price - entry_price) * num_shares
        trades_list.append({
            'ticker': entry_ticker,
            'entry_date': entry_date,
            'entry_price': entry_price,
            'exit_date': exit_date,
            'exit_price': exit_price,
            'num_shares': num_shares,
            'profit': profit,
            'holding_days': (exit_date - entry_date).days,
            'entry_score': entry_score,
            'exit_score': exit_row['score'] # This is the score of the *last recommended ticker* on the last day.
        })

df_trades = pd.DataFrame(trades_list)

print("\nCalculated Trades and Profits:")
display(df_trades.head(10))
print(f"\nTotal Number of Trades: {len(df_trades)}")
print(f"Total Profit/Loss: ${df_trades['profit'].sum():,.2f}")


Calculated Trades and Profits:


,ticker,entry_date,entry_price,exit_date,exit_price,num_shares,profit,holding_days,entry_score,exit_score
0,SH,2020-01-27,95.599998,2020-01-28,94.639999,52.0,-49.919952,1,3,3
1,QQQ,2020-01-28,221.449997,2020-01-31,219.070007,22.0,-52.359772,3,3,3
2,SH,2020-01-31,96.199997,2020-02-04,93.959999,51.0,-114.239891,4,3,4
3,QQQ,2020-02-04,227.470001,2020-02-24,221.389999,21.0,-127.680038,20,4,3
4,PSQ,2020-02-24,117.650002,2020-03-03,123.699997,42.0,254.099808,8,3,4
5,SH,2020-03-03,102.519997,2020-03-26,109.599998,48.0,339.840088,23,4,3
6,QQQ,2020-03-26,191.899994,2020-04-01,182.309998,26.0,-249.339905,6,3,3
7,SH,2020-04-01,115.720001,2020-04-06,106.800003,43.0,-383.559921,5,3,4
8,QQQ,2020-04-06,196.479996,2020-04-21,204.889999,25.0,210.250092,15,4,3
9,SPY,2020-04-21,273.040009,2020-04-24,282.970001,18.0,178.739868,3,3,4



Total Number of Trades: 261
Total Profit/Loss: $-2,141.59


In [132]:
# Calculate Yearly Profit Summary and Max Drawdown

df_trades['year'] = df_trades['entry_date'].dt.year
yearly_profit_summary = df_trades.groupby('year')['profit'].sum().reset_index()

investment_amount = 5000 # As specified by the user
yearly_profit_summary['percent_return'] = (yearly_profit_summary['profit'] / investment_amount) * 100

def calculate_max_drawdown(equity_curve):
    """
    Calculates the maximum drawdown of an equity curve.
    Handles cases where equity_curve can be zero or negative.

    Args:
        equity_curve (pd.Series): A Series of cumulative portfolio values.

    Returns:
        float: The maximum drawdown as a positive percentage. Returns 0.0 if no drawdown.
    """
    if equity_curve.empty or equity_curve.isnull().all() or equity_curve.nunique() <= 1:
        return 0.0

    equity_curve = pd.to_numeric(equity_curve, errors='coerce').fillna(0)
    peak = equity_curve.expanding(min_periods=1).max()

    max_dd = 0.0
    for i in range(len(equity_curve)):
        current_peak = peak.iloc[i]
        current_value = equity_curve.iloc[i]

        if current_peak > 0:
            drawdown = (current_peak - current_value) / current_peak
            max_dd = max(max_dd, drawdown)
        elif current_peak == 0 and current_value < 0:
            # If peak is 0 and value drops below 0, it's 100% drawdown relative to starting point of 0
            max_dd = max(max_dd, 1.0) # 100%

    return max_dd * 100 # Return as positive percentage

# Generate a daily profit series for drawdown calculation
min_date = df_trades['entry_date'].min()
max_date = df_trades['exit_date'].max()
all_dates = pd.date_range(start=min_date, end=max_date, freq='D')

daily_profits_series = pd.Series(0.0, index=all_dates)

# Assign profits to exit dates. If multiple trades exit on the same day, sum them.
for index, trade in df_trades.iterrows():
    # Use .loc for setting value by index, handling potential multiple exits on same day
    daily_profits_series.loc[trade['exit_date']] += trade['profit']

# Calculate yearly max drawdowns
yearly_max_drawdowns = []

for year in yearly_profit_summary['year'].unique():
    year_start = pd.Timestamp(f'{year}-01-01')
    year_end = pd.Timestamp(f'{year}-12-31')

    # Filter daily profits for the current year
    # Use .copy() to avoid SettingWithCopyWarning if modifying later
    daily_profits_this_year = daily_profits_series.loc[year_start:year_end].copy()

    if not daily_profits_this_year.empty:
        # Create an equity curve for the year, starting with the initial investment amount
        # and accumulating daily profits.
        yearly_equity_curve = pd.Series(investment_amount, index=daily_profits_this_year.index)
        yearly_equity_curve = yearly_equity_curve + daily_profits_this_year.cumsum()
        max_dd = calculate_max_drawdown(yearly_equity_curve)
    else:
        max_dd = 0.0 # No trades for the year

    yearly_max_drawdowns.append({'year': year, 'max_drawdown_percent': max_dd})

df_yearly_max_drawdowns = pd.DataFrame(yearly_max_drawdowns)
# Ensure 'year' column is set as index or used for merge key
df_yearly_max_drawdowns.set_index('year', inplace=True)

# Merge max drawdowns into the main yearly profit summary
yearly_profit_summary = yearly_profit_summary.merge(df_yearly_max_drawdowns, on='year', how='left')

print("\nYearly Profit Summary with Percentage Return and Max Drawdown:")
display(yearly_profit_summary.round(2))


Yearly Profit Summary with Percentage Return and Max Drawdown:


,year,profit,percent_return,max_drawdown_percent
0,2020,463.09,9.26,12.88
1,2021,-450.51,-9.01,11.89
2,2022,-1217.18,-24.34,29.22
3,2023,812.55,16.25,6.86
4,2024,-820.30,-16.41,21.40
5,2025,-929.24,-18.58,20.92
